In [45]:
import pandas as pd
import numpy as np
from pypdf import PdfReader
from rank_bm25 import BM25Okapi


In [46]:
d1=PdfReader("test.pdf")
text1 = ""
for page in d1.pages:
    text1 += page.extract_text()

d2=PdfReader("learning-langchain-for-true-epub-9781098167288.pdf")
text2 = ""
for page in d2.pages:
    text2 += page.extract_text()

d3=PdfReader("Natural-Language-Processing-Python (1).pdf")
text3 = ""
for page in d3.pages:
    text3 += page.extract_text()

In [47]:
corpus=[text1,
        text2,
        text3
        ]

In [53]:
query1="what is tokenization"
query2="what are different types of tokenizer"

In [54]:
tokenizer_corpus=[doc.lower().split() for doc in corpus]
tokenizer_query1=query1.lower().split()
tokenizer_query2=query2.lower().split()

In [55]:
bm25=BM25Okapi(corpus)
scores=bm25.get_scores(tokenizer_query2)

In [56]:
for i,s in enumerate(scores):
    print(f"Doc {i+1} -> score {s:3f} | {corpus[i]}")

Doc 1 -> score 0.000000 |  Praise for Hands-On Large Language Models
This is an exceptional guide to the world of language models and their
practical applications in industry. Its highly-visual coverage of
generative, representational, and retrieval applications of language
models empowers readers to quickly understand, use, and refine LLMs.
Highly recommended!
—Nils Reimers, Director of Machine Learning at Cohere |
creator of sentence-transformers
Jay and Maarten have continued their tradition of providing beautifully
illustrated and insightful descriptions of complex topics in their new
book. Bolstered with working code, timelines, and references to key
papers, their book is a valuable resource for anyone looking to
understand the main techniques behind how Large Language Models are
built.
—Andrew Ng, founder of DeepLearning.AI
I can’t think of another book that is more important to read right now. On
every single page, I learned something that is critical to success in this
era of l

In [ ]:
import re
import pandas as pd
import numpy as np
from pypdf import PdfReader
from rank_bm25 import BM25Okapi

def extract_text(path):
    reader = PdfReader(path)
    return " ".join(page.extract_text() or "" for page in reader.pages)

corpus = [
    extract_text("test.pdf"),
    extract_text("learning-langchain-for-true-epub-9781098167288.pdf"),
    extract_text("Natural-Language-Processing-Python (1).pdf"),
]

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

tokenized_corpus = [tokenize(doc) for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

queries = {
    "what is tokenization": tokenize("what is tokenization"),
    "what are different types of tokenizer": tokenize("what are different types of tokenizer"),
}

for query_text, query_tokens in queries.items():
    print(f"Query: '{query_text}'")
    print("-" * 60)
    scores = bm25.get_scores(query_tokens)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    for rank, (i, score) in enumerate(ranked, 1):
        print(f"  Rank {rank}: Doc {i+1}  |  Score: {score:.4f}")
        print(f"  Preview: {corpus[i][:200].strip()}...")
        print()


Query: 'what is tokenization'
------------------------------------------------------------
  Rank 1: Doc 1  |  Score: 0.0841
  Preview: Praise for Hands-On Large Language Models
This is an exceptional guide to the world of language models and their
practical applications in industry. Its highly-visual coverage of
generative, represen...

  Rank 2: Doc 3  |  Score: 0.0833
  Preview: MANNING
Hobson Lane
Cole Howard
Hannes Max Hapke
Foreword by Dr. Arwen Griffioen
Understanding, analyzing, and generating text with Python
IN ACTION Chatbot Recirculating (Recurrent) Pipeline
Database...

  Rank 3: Doc 2  |  Score: 0.0707
  Preview: Praise	for	
Learning	LangChain
With	clear	explanations	and	actionable	techniques,	this	is	the	go-to
resource	for	anyone	looking	to	harness	LangChain’s	power	for	production-
ready	generative	AI	and	ag...

Query: 'what are different types of tokenizer'
------------------------------------------------------------
  Rank 1: Doc 1  |  Score: 0.1678
  Preview: Praise f

In [57]:
import re
from pypdf import PdfReader
from rank_bm25 import BM25Okapi

# Extract Text from PDF
def extract_text(path):
    reader = PdfReader(path)
    return " ".join(page.extract_text() or "" for page in reader.pages)

# Chunking Function (300 words per chunk, 50 word overlap)
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Load & Chunk PDFs
pdf_files = [
    "test.pdf",
    "learning-langchain-for-true-epub-9781098167288.pdf",
    "Natural-Language-Processing-Python (1).pdf",
]

all_chunks = []       # stores the raw chunk text
chunk_sources = []    # tracks which PDF each chunk came from

for pdf in pdf_files:
    text = extract_text(pdf)
    chunks = chunk_text(text, chunk_size=300, overlap=50)
    all_chunks.extend(chunks)
    chunk_sources.extend([pdf] * len(chunks))
    print(f"{pdf} -> {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

# Tokenizer
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

tokenized_chunks = [tokenize(chunk) for chunk in all_chunks]

# BM25 Setup
bm25 = BM25Okapi(tokenized_chunks)

# Search Function
def search(query, top_k=3):
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
    print(f"Query: '{query}'")
    print("-" * 60)
    for rank, (i, score) in enumerate(ranked, 1):
        print(f"  Rank {rank} | Score: {score:.4f} | Source: {chunk_sources[i]}")
        print(f"  {all_chunks[i][:250].strip()}...")
        print()

# Run Queries
search("what is tokenization")
search("what are different types of tokenizer")

test.pdf -> 381 chunks
learning-langchain-for-true-epub-9781098167288.pdf -> 294 chunks
Natural-Language-Processing-Python (1).pdf -> 822 chunks

Total chunks: 1497
Query: 'what is tokenization'
------------------------------------------------------------
  Rank 1 | Score: 8.7738 | Source: test.pdf
  shows. Figure 2-5. Tokenizers are also used to process the output of the model by converting the output token ID into the word or token associated with that ID. Word Versus Subword Versus Character Versus Byte Tokens The tokenization scheme we just d...

  Rank 2 | Score: 8.2832 | Source: test.pdf
  AI encompasses many developments and models aiming to represent and generate language, as illustrated in Figure 1-1. Figure 1-1. A peek into the history of Language AI. Language, however, is a tricky concept for computers. Text is unstructured in nat...

  Rank 3 | Score: 7.8521 | Source: test.pdf
  a + b This may be suboptimal for a code-focused model. Code-focused models are often improved by